# Signal vs Noise Assessment of GWAS Associations Using Dosage-Based Analysis

This notebook evaluates whether GWAS-identified associations represent true genetic effects or are likely driven by noise by directly analyzing phenotype variation across genotype dosage levels.

Significant loci are first selected from GWAS results based on a defined p-value threshold and mapped to TAGLO-based genotype data. Phenotype data are then reshaped into a long format and integrated with genotype dosage information to construct a complete dataset covering all varieties and loci.

For each trait–locus combination, phenotype values are summarized across discrete dosage groups. The analysis then assesses whether phenotype changes consistently with genotype dosage, using criteria such as the number of dosage groups, sample size per group, and variability of phenotype across groups.

Loci are classified as **"signal"** when they show structured and reproducible phenotype differences across dosage levels, indicating a likely genetic effect. Otherwise, they are classified as **"noise"**, suggesting that the observed association may not reflect a true biological relationship.

This approach provides a robust validation framework for GWAS results by testing whether genotype dosage translates into meaningful phenotypic variation.


In [0]:
import yaml
import pyspark.sql.functions as F
from pyspark.sql.types import NumericType
from IPython.display import display
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from pyspark.sql import types as T
from pyspark.sql import Window as W
from pyspark.sql import SparkSession


In [0]:
# ============================================================
# 1) CONFIG & SETTINGS
# ============================================================

from pyspark.sql.types import NumericType

CONFIG_PATH = "/Volumes/bmqg/default_bronze/fatemeh/config_mixed.yaml"

with open(CONFIG_PATH, "r") as f:
    CONFIG = yaml.safe_load(f)

# paths
GWAS_TABLE = CONFIG["data"]["gwas_table"]
PATHS = CONFIG["paths"]

PHENO_PATH = PATHS["aroma_matrix"]
TAGLO_TABLE = PATHS["TAGLO_TABLE"]

# parameters
P_THRESH = 1e-6
MIN_GROUPS = 3
MIN_N_PER_GROUP = 3

print("GWAS table :", GWAS_TABLE)
print("TAGLO table:", TAGLO_TABLE)
print("PHENO path :", PHENO_PATH)

GWAS table : bmqg.gwas.run_local_20251207
TAGLO table: bmqg.default_silver.taglotype_silver
PHENO path : /Volumes/bmqg/default_bronze/fatemeh/final_project/csv_outputs/aroma_matrix_GWAS_clean.csv


# LOAD & PREPARE GWAS

In [0]:
# -----------------------------
# 1) LOAD & FILTER GWAS
# -----------------------------
gwas_raw = (
    spark.table(GWAS_TABLE)
    .filter(F.col("p_wald") <= F.lit(P_THRESH))
    .select(
        F.col("trait").cast("string").alias("trait"),
        F.col("chrom").cast("string").alias("chrom"),
        F.col("start").cast("long").alias("ps"),
        F.col("taglo_id1").cast("long"),
        F.col("taglo_id2").cast("long"),
        F.col("taglo_id3").cast("long"),
        F.col("taglo_id4").cast("long"),
    )
    .filter(F.col("ps") > 0)
)

# -----------------------------
# 2) UNPIVOT taglo_id1..4 → taglo_id
#    then COLLAPSE to one row per (trait, chrom, taglo_id)
#    to avoid duplicate counting from multiple ps hits.
# -----------------------------
gwas_taglo = (
    gwas_raw
    .withColumn("taglo_id", F.explode(F.array("taglo_id1","taglo_id2","taglo_id3","taglo_id4")))
    .filter(F.col("taglo_id").isNotNull())
    .filter(F.col("taglo_id") > 0)
    .select("trait","chrom","ps","taglo_id")
    .groupBy("trait","chrom","taglo_id")
    .agg(F.min("ps").alias("start"))
)

print("GWAS taglo rows:", gwas_taglo.count())
display(gwas_taglo.limit(10))


GWAS taglo rows: 12701


DataFrame[trait: string, chrom: string, taglo_id: bigint, start: bigint]

# LOAD & CLEAN TAGLO (DOSAGE)

In [0]:
# -----------------------------
# 3) LOAD TAGLO TABLE (REAL DOSAGE)

# -----------------------------
taglo_silver = (
    spark.table(TAGLO_TABLE)
    .select(
        F.col("VARIETY").cast("string").alias("VARIETY"),
        F.col("taglo_id").cast("long").alias("taglo_id"),
        F.col("value").cast("double").alias("dosage_raw"),
    )
    .dropna(subset=["VARIETY","taglo_id"])
    .groupBy("VARIETY","taglo_id")
    .agg(F.max("dosage_raw").alias("dosage_raw"))
)

taglo_silver = (
    taglo_silver
    .withColumn("dosage_int", F.round(F.col("dosage_raw")).cast("int"))
    .withColumn("dosage_int", F.when(F.col("dosage_int").isNull(), F.lit(0)).otherwise(F.col("dosage_int")))
    .withColumn("dosage_int", F.when(F.col("dosage_int") < 0, F.lit(0)).otherwise(F.col("dosage_int")))
    .withColumn("dosage_int", F.when(F.col("dosage_int") > 4, F.lit(4)).otherwise(F.col("dosage_int")))
    .select("VARIETY","taglo_id", F.col("dosage_int").alias("dosage"))
)

print("Taglo dosage rows:", taglo_silver.count())
display(taglo_silver.limit(10))


Taglo dosage rows: 58052698


DataFrame[VARIETY: string, taglo_id: bigint, dosage: int]

#LOAD PHENOTYPE

In [0]:
# ============================================================
# 4) PHENOTYPE → LONG FORMAT
# ============================================================

pheno_wide = spark.read.csv(PHENO_PATH, header=True, inferSchema=True)

numeric_traits = [
    f.name for f in pheno_wide.schema.fields
    if f.name.upper() != "VARIETY" and isinstance(f.dataType, NumericType)
]

print("Numeric traits:", len(numeric_traits))

pheno_long = pheno_wide.select(
    F.col("VARIETY"),
    F.expr(
        f"stack({len(numeric_traits)}, " +
        ",".join([f"'{c}', `{c}`" for c in numeric_traits]) +
        ") as (trait, trait_value)"
    )
).dropna()

Numeric traits: 67


# BUILD COMPLETE DOSAGE GRID

In [0]:
# -----------------------------
# 5) BUILD COMPLETE (VARIETY × GWAS taglo_id) GRID
#    so missing genotypes become dosage=0 (critical fix)
# -----------------------------
varieties = pheno_long.select("VARIETY").distinct()

taglo_ids = gwas_taglo.select("chrom","taglo_id").distinct()

all_pairs = varieties.crossJoin(taglo_ids)

dosage_complete = (
    all_pairs
    .join(taglo_silver, on=["VARIETY","taglo_id"], how="left")
    .withColumn("dosage", F.coalesce(F.col("dosage"), F.lit(0)))
)

print("Complete dosage grid rows:", dosage_complete.count())
display(dosage_complete.limit(10))


Complete dosage grid rows: 404482


DataFrame[VARIETY: string, taglo_id: bigint, chrom: string, dosage: int]

# MERGE ALL DATA

In [0]:
# -----------------------------
# 6) COMBINE GWAS + DOSAGE + PHENOTYPE
# -----------------------------
full = (
    gwas_taglo
    .join(dosage_complete, on=["chrom","taglo_id"], how="inner")
    .join(pheno_long, on=["VARIETY","trait"], how="inner")
    .select("trait","chrom","start","taglo_id","VARIETY","dosage","trait_value")
)

print("Full rows:", full.count())
display(full.limit(10))


Full rows: 1193894


DataFrame[trait: string, chrom: string, start: bigint, taglo_id: bigint, VARIETY: string, dosage: int, trait_value: double]

# DOSAGE-LEVEL STATS

In [0]:
# -----------------------------
# 7) DOSAGE-LEVEL STATISTICS
# -----------------------------
dosage_stats = (
    full
    .groupBy("trait","chrom","taglo_id","dosage")
    .agg(
        F.count("*").alias("n_samples"),
        F.mean("trait_value").alias("mean_trait")
    )
)


# SIGNAL vs NOISE CLASSIFICATION

In [0]:

# -----------------------------
# 8) SIGNAL vs NOISE DECISION (basic + defensible)
# -----------------------------
signal_table = (
    dosage_stats
    .groupBy("trait","chrom","taglo_id")
    .agg(
        F.min("n_samples").alias("min_n_per_group"),
        F.countDistinct("dosage").alias("n_dosage_groups"),
        (F.max("mean_trait") - F.min("mean_trait")).alias("delta_extreme"),
        F.stddev("mean_trait").alias("between_dosage_sd"),
    )
    .withColumn(
        "signal_type",
        F.when(
            (F.col("n_dosage_groups") >= F.lit(MIN_GROUPS)) &
            (F.col("min_n_per_group") >= F.lit(MIN_N_PER_GROUP)) &
            (F.col("between_dosage_sd").isNotNull()) &
            (F.col("between_dosage_sd") > 0) &
            (F.col("delta_extreme") > 0),
            F.lit("signal")
        ).otherwise(F.lit("noise"))
    )
)

# -----------------------------
# 

In [0]:
# 9) FINAL OUTPUT (WITH START)
# -----------------------------
final_result = (
    signal_table
    .join(
        gwas_taglo.select("trait","chrom","taglo_id","start"),
        on=["trait","chrom","taglo_id"],
        how="left"
    )
    .select(
        "trait","chrom","start","taglo_id","signal_type",
        "n_dosage_groups","min_n_per_group","delta_extreme","between_dosage_sd"
    )
    .orderBy("signal_type","chrom","start","trait")
)

display(final_result)


DataFrame[trait: string, chrom: string, start: bigint, taglo_id: bigint, signal_type: string, n_dosage_groups: bigint, min_n_per_group: bigint, delta_extreme: double, between_dosage_sd: double]

In [0]:
# -----------------------------
# 10) SAVE AS SINGLE CSV
# -----------------------------
OUTPUT_DIR = "/Volumes/bmqg/default_bronze/fatemeh/final_project/csv_outputs/tmp_signal_noise"
FINAL_PATH = "/Volumes/bmqg/default_bronze/fatemeh/final_project/csv_outputs/signal_noise.csv"

(
    final_result
    .coalesce(1)
    .write
    .mode("overwrite")
    .option("header", True)
    .csv(OUTPUT_DIR)
)

print("Spark wrote a folder to:", OUTPUT_DIR)


Spark wrote a folder to: /Volumes/bmqg/default_bronze/fatemeh/final_project/csv_outputs/tmp_signal_noise
